# 04b — Temporal Attention ile Koalisyon Tahmin Modeli

**Motivasyon:** Mevcut model her yılı bağımsız encode ediyor — bir ülkenin t yılındaki embedding'i sadece t yılının grafına bağımlı. Ama koalisyonlar zamanda gelişir: ittifaklar güçlenir/zayıflar, ticaret ilişkileri değişir.

**Yenilik:** RGCN çıktılarına bir **Temporal Multi-Head Attention** katmanı ekliyoruz:
- Her ülke için son W yılın embedding'lerini topluyoruz
- Multi-head attention ile zamansal ağırlıklı temsil hesaplıyoruz
- Bu sayede model "ilişkilerin evrimi"ni yakalıyor

**Beklenti:** Test AUC'yi 0.82 → 0.85+ seviyesine çıkarmak (temporal shift'e karşı daha robust)

**Mimari:**
```
z_i^t     (RGCN) ─┐
z_i^{t-1} (RGCN) ─┼─→ MultiHead Temporal Attention → z̃_i^t (enriched)
z_i^{t-2} (RGCN) ─┤
...               ─┘
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

START_YEAR = 1946
TRAIN_END = 1999
VAL_END = 2009
END_YEAR = 2016

In [ ]:
!pip install -q torch torch_geometric numpy pandas tqdm scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Veri Yükleme (04_model ile aynı)

In [ ]:
# Koalisyon örneklerini yükle
pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

print(f'Pozitif örnekler: {len(pos_df)}')
print(f'Negatif örnekler: {len(neg_df)}')

In [ ]:
# Snapshot'ları yükle — TÜM yıllar (temporal window için gerekli)
EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = len(EDGE_TYPES)

def load_snapshot(year):
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    data = torch.load(path, weights_only=False)
    x = data['country'].x
    cow_codes = data['country'].cow_code.numpy()
    
    edge_index_list = []
    edge_type_list = []
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            edge_index_list.append(ei)
            edge_type_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
    
    if not edge_index_list:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_type = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(edge_index_list, dim=1)
        edge_type = torch.cat(edge_type_list)
    
    return {
        'x': x, 'edge_index': edge_index, 'edge_type': edge_type,
        'cow_codes': cow_codes,
        'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)},
        'year': year,
    }

# Tüm yılların snapshot'larını yükle (temporal window için hepsine ihtiyaç var)
snapshots = {}
for year in range(START_YEAR, END_YEAR + 1):
    snap = load_snapshot(year)
    if snap is not None:
        snapshots[year] = snap

print(f'Yüklenen snapshot sayısı: {len(snapshots)}')
NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Düğüm özellik boyutu: {NUM_FEATURES}')

In [ ]:
# Örnekleri hazırla (04_model ile aynı)
def parse_members(members_str):
    if isinstance(members_str, list):
        return [int(x) for x in members_str]
    if isinstance(members_str, str):
        return [int(x) for x in members_str.split(',')]
    return []

def build_samples(df, label):
    samples = []
    for _, row in df.iterrows():
        year = int(row['year'])
        if year not in snapshots:
            continue
        members = parse_members(row['members_str'])
        code_to_idx = snapshots[year]['code_to_idx']
        valid_members = [m for m in members if m in code_to_idx]
        if len(valid_members) < 2:
            continue
        sample = {
            'year': year,
            'members': valid_members,
            'member_idx': [code_to_idx[m] for m in valid_members],
            'label': label,
            'duration': float(row.get('total_duration', 0)) if label == 1 else 0.0,
            'cohesion': float(row.get('internal_vote_agreement', 0) or 0) if label == 1 else 0.0,
        }
        samples.append(sample)
    return samples

pos_samples = build_samples(pos_df, label=1)
neg_samples = build_samples(neg_df, label=0)
all_samples = pos_samples + neg_samples

train_samples = [s for s in all_samples if s['year'] <= TRAIN_END]
val_samples = [s for s in all_samples if TRAIN_END < s['year'] <= VAL_END]
test_samples = [s for s in all_samples if s['year'] > VAL_END]

print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')
print(f'Train pos/neg: {sum(1 for s in train_samples if s["label"]==1)}/{sum(1 for s in train_samples if s["label"]==0)}')

## 2. Model Tanımı — Temporal Attention

In [ ]:
class RGCNEncoder(nn.Module):
    """2-katmanlı RGCN encoder (04a ile aynı mimari)."""
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_relations, dropout=0.2):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    
    def forward(self, x, edge_index, edge_type):
        h = self.conv1(x, edge_index, edge_type)
        h = self.norm1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, edge_type)
        h = self.norm2(h)
        return h


class TemporalAttention(nn.Module):
    """Multi-head temporal attention.
    
    Her ülke i için, son W yılın embedding'lerine attention uygular:
    z̃_i^t = Attention(Q=z_i^t, K=V=[z_i^{t-W+1}, ..., z_i^t])
    
    Temporal positional encoding ile yıl sırası bilgisi de verilir.
    """
    def __init__(self, embed_dim, num_heads=4, window_size=5, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.window_size = window_size
        self.num_heads = num_heads
        
        # Multi-head attention
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        
        # Temporal positional encoding (learnable)
        self.temporal_pe = nn.Parameter(torch.randn(window_size, embed_dim) * 0.02)
        
        # Layer norm + residual
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, z_current, z_history_list):
        """Temporal attention uygula.
        
        Args:
            z_current: (N, D) — mevcut yılın embedding'leri
            z_history_list: list of (N, D) — geçmiş yılların embedding'leri
                           En eski → en yeni sıralı, son eleman z_current
                           Eksik yıllar None olabilir
        
        Returns:
            z_enriched: (N, D) — temporal-enriched embedding'ler
        """
        N, D = z_current.shape
        W = len(z_history_list)
        
        # Geçmiş embedding'leri stack et, None olanları sıfır ile doldur
        history_stack = []
        mask = []  # True = padded (ignore)
        for z in z_history_list:
            if z is not None:
                history_stack.append(z)
                mask.append(False)
            else:
                history_stack.append(torch.zeros(N, D, device=z_current.device))
                mask.append(True)
        
        # (N, W, D)
        temporal_seq = torch.stack(history_stack, dim=1)
        
        # Temporal positional encoding ekle
        pe = self.temporal_pe[:W].unsqueeze(0)  # (1, W, D)
        temporal_seq = temporal_seq + pe
        
        # Attention mask (padded pozisyonları maskele)
        key_padding_mask = torch.tensor(
            mask, dtype=torch.bool, device=z_current.device
        ).unsqueeze(0).expand(N, -1)  # (N, W)
        
        # Query = mevcut yıl, Key/Value = temporal sequence
        query = z_current.unsqueeze(1)  # (N, 1, D)
        
        # Multi-head attention
        attn_out, _ = self.attention(
            query=query,
            key=temporal_seq,
            value=temporal_seq,
            key_padding_mask=key_padding_mask,
        )  # (N, 1, D)
        
        attn_out = attn_out.squeeze(1)  # (N, D)
        
        # Residual connection + layer norm
        z_enriched = self.norm(z_current + self.dropout(attn_out))
        
        return z_enriched


class CoalitionHead(nn.Module):
    """Koalisyon skorlama başlığı: DeepSets + pairwise bilinear."""
    def __init__(self, embed_dim, hidden_dim=64, lambda_pair=0.5):
        super().__init__()
        self.lambda_pair = lambda_pair
        self.set_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )
        self.bilinear = nn.Bilinear(embed_dim, embed_dim, 1)
        self.duration_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Softplus(),
        )
        self.cohesion_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Sigmoid(),
        )
    
    def forward(self, z_members):
        K = z_members.shape[0]
        z_pool = z_members.mean(dim=0, keepdim=True)
        set_score = self.set_mlp(z_pool).squeeze()
        
        if K >= 2:
            idx_i, idx_j = torch.triu_indices(K, K, offset=1, device=z_members.device)
            z_i = z_members[idx_i]
            z_j = z_members[idx_j]
            pair_scores = self.bilinear(z_i, z_j).squeeze(-1)
            pair_mean = pair_scores.mean()
        else:
            pair_mean = torch.tensor(0.0, device=z_members.device)
        
        score = set_score + self.lambda_pair * pair_mean
        duration_pred = self.duration_head(z_pool).squeeze()
        cohesion_pred = self.cohesion_head(z_pool).squeeze()
        return score, duration_pred, cohesion_pred


class TemporalCoalitionGNN(nn.Module):
    """Tam model: RGCN encoder + Temporal Attention + Coalition Head."""
    def __init__(self, num_features, hidden_dim, embed_dim, num_relations,
                 head_hidden=64, dropout=0.2, temporal_window=5, num_heads=4):
        super().__init__()
        self.temporal_window = temporal_window
        
        self.encoder = RGCNEncoder(
            in_channels=num_features,
            hidden_channels=hidden_dim,
            out_channels=embed_dim,
            num_relations=num_relations,
            dropout=dropout,
        )
        
        self.temporal_attn = TemporalAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=temporal_window,
            dropout=0.1,
        )
        
        self.head = CoalitionHead(embed_dim, hidden_dim=head_hidden)
    
    def encode_graph(self, x, edge_index, edge_type):
        """Tek bir yılı encode et (RGCN only, temporal yok)."""
        return self.encoder(x, edge_index, edge_type)
    
    def encode_temporal(self, year, snapshots, device, raw_embeddings=None):
        """Temporal attention ile zenginleştirilmiş embedding hesapla.
        
        Args:
            year: hedef yıl
            snapshots: tüm snapshot dict
            device: torch device
            raw_embeddings: önceden hesaplanmış RGCN çıktıları (cache)
        
        Returns:
            z_enriched: (N, D) — temporal-enriched embedding'ler
        """
        # Mevcut yılın düğüm kümesini referans al
        snap_t = snapshots[year]
        cow_codes_t = snap_t['cow_codes']
        N = len(cow_codes_t)
        code_to_idx_t = snap_t['code_to_idx']
        
        # Mevcut yılın RGCN embedding'i
        if raw_embeddings and year in raw_embeddings:
            z_current = raw_embeddings[year]
        else:
            x = snap_t['x'].to(device)
            ei = snap_t['edge_index'].to(device)
            et = snap_t['edge_type'].to(device)
            z_current = self.encode_graph(x, ei, et)
        
        # Temporal window: son W yılın embedding'lerini topla
        D = z_current.shape[1]
        z_history = []
        
        for offset in range(self.temporal_window - 1, -1, -1):  # W-1, W-2, ..., 0
            past_year = year - offset
            
            if past_year not in snapshots:
                z_history.append(None)
                continue
            
            # Geçmiş yılın embedding'i
            if raw_embeddings and past_year in raw_embeddings:
                z_past_full = raw_embeddings[past_year]
            else:
                snap_past = snapshots[past_year]
                x_p = snap_past['x'].to(device)
                ei_p = snap_past['edge_index'].to(device)
                et_p = snap_past['edge_type'].to(device)
                z_past_full = self.encode_graph(x_p, ei_p, et_p)
            
            # Geçmiş yıldan mevcut yılın ülkelerine eşleme
            snap_past = snapshots[past_year]
            code_to_idx_past = snap_past['code_to_idx']
            
            z_aligned = torch.zeros(N, D, device=device)
            for i, cow in enumerate(cow_codes_t):
                cow_int = int(cow)
                if cow_int in code_to_idx_past:
                    past_idx = code_to_idx_past[cow_int]
                    z_aligned[i] = z_past_full[past_idx]
                # else: sıfır kalır (ülke o yıl yok)
            
            z_history.append(z_aligned)
        
        # Temporal attention uygula
        z_enriched = self.temporal_attn(z_current, z_history)
        return z_enriched
    
    def score_coalition(self, z_all, member_idx):
        """Embedding'lerden koalisyon skorla."""
        z_members = z_all[member_idx]
        return self.head(z_members)


# Model oluştur
HIDDEN_DIM = 128
EMBED_DIM = 128
HEAD_HIDDEN = 64
DROPOUT = 0.2
TEMPORAL_WINDOW = 5  # Son 5 yıl
NUM_HEADS = 4

model = TemporalCoalitionGNN(
    num_features=NUM_FEATURES,
    hidden_dim=HIDDEN_DIM,
    embed_dim=EMBED_DIM,
    num_relations=NUM_RELATIONS,
    head_hidden=HEAD_HIDDEN,
    dropout=DROPOUT,
    temporal_window=TEMPORAL_WINDOW,
    num_heads=NUM_HEADS,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parametreleri: {n_params:,}')
print(f'  (Temporal attention eklenmesiyle ~{n_params - 134596:,} parametre artışı)')
print(f'\nTemporal window: {TEMPORAL_WINDOW} yıl')
print(f'Attention heads: {NUM_HEADS}')
print(f'\nModel yapısı:\n{model}')

In [ ]:
# Pretrained encoder ağırlıklarını yükle (04a'dan)
pretrain_path = os.path.join(MODEL_DIR, 'encoder_pretrained.pt')

if os.path.exists(pretrain_path):
    checkpoint = torch.load(pretrain_path, weights_only=False)
    model.encoder.load_state_dict(checkpoint['encoder_state_dict'])
    print(f'✅ Pretrained encoder yüklendi')
    print(f'   Pretrain epoch: {checkpoint["pretrain_epoch"]}, val_loss: {checkpoint["pretrain_val_loss"]:.4f}')
else:
    print('⚠️  Pretrained encoder bulunamadı')

## 3. Eğitim Döngüsü

**Faz 1** (10 epoch): Encoder frozen, temporal attention + head eğitilir  
**Faz 2** (190 epoch): End-to-end, differential LR

In [ ]:
# Hiperparametreler
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 190
TOTAL_EPOCHS = PHASE1_EPOCHS + PHASE2_EPOCHS

HEAD_LR = 1e-3
TEMPORAL_LR = 5e-4  # Temporal attention LR (head ile encoder arası)
ENCODER_LR = 1e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 30
POS_WEIGHT = 2.5

LAMBDA_MAIN = 1.0
LAMBDA_DURATION = 0.3
LAMBDA_COHESION = 0.3

BATCH_SIZE = 64

# Faz 1: Encoder frozen, temporal + head eğitilir
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam([
    {'params': model.temporal_attn.parameters(), 'lr': TEMPORAL_LR},
    {'params': model.head.parameters(), 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Faz 1: Encoder frozen, temporal+head eğitilir ({PHASE1_EPOCHS} epoch)')
print(f'  Eğitilebilir parametre: {n_trainable:,}')
print(f'Faz 2: End-to-end ({PHASE2_EPOCHS} epoch, patience={PATIENCE})')
print(f'POS_WEIGHT: {POS_WEIGHT}')

In [ ]:
# === HIZLANDIRMA: Önceden hesaplanmış eşleme haritaları ===

# COW-bazlı ülke eşleme haritalarını BİR KEZ hesapla (epoch'lar arası değişmez)
# alignment_maps[year][past_year] = (target_indices, source_indices)
# → z_aligned[target_indices] = z_past[source_indices]

print('Alignment haritaları hesaplanıyor...')
alignment_maps = {}
for year in tqdm(snapshots.keys(), desc='Alignment maps'):
    alignment_maps[year] = {}
    cow_codes_t = snapshots[year]['cow_codes']
    code_to_idx_t = snapshots[year]['code_to_idx']
    
    for offset in range(TEMPORAL_WINDOW):
        past_year = year - offset
        if past_year not in snapshots:
            continue
        code_to_idx_past = snapshots[past_year]['code_to_idx']
        
        # Ortak ülkeleri bul
        target_idx = []
        source_idx = []
        for i, cow in enumerate(cow_codes_t):
            cow_int = int(cow)
            if cow_int in code_to_idx_past:
                target_idx.append(i)
                source_idx.append(code_to_idx_past[cow_int])
        
        alignment_maps[year][past_year] = (
            torch.tensor(target_idx, dtype=torch.long),
            torch.tensor(source_idx, dtype=torch.long),
        )

print(f'✅ {len(alignment_maps)} yıl için alignment haritaları hazır')


def encode_all_years_raw(model, snapshots, device, no_grad=False):
    """Tüm yılların RAW RGCN embedding'lerini hesapla."""
    embeddings = {}
    ctx = torch.no_grad() if no_grad else torch.enable_grad()
    with ctx:
        for year, snap in snapshots.items():
            x = snap['x'].to(device)
            ei = snap['edge_index'].to(device)
            et = snap['edge_type'].to(device)
            z = model.encode_graph(x, ei, et)
            embeddings[year] = z
    return embeddings


def encode_temporal_fast(model, year, snapshots, device, raw_embeddings, alignment_maps):
    """Hızlı temporal encoding — önceden hesaplanmış raw embeddings ve alignment kullanır."""
    snap_t = snapshots[year]
    N = len(snap_t['cow_codes'])
    D = raw_embeddings[year].shape[1]
    z_current = raw_embeddings[year]
    
    z_history = []
    for offset in range(model.temporal_window - 1, -1, -1):  # W-1, ..., 0
        past_year = year - offset
        
        if past_year not in raw_embeddings:
            z_history.append(None)
            continue
        
        z_past = raw_embeddings[past_year]
        
        # Önceden hesaplanmış alignment ile vektörize eşleme
        if past_year in alignment_maps[year]:
            tgt_idx, src_idx = alignment_maps[year][past_year]
            z_aligned = torch.zeros(N, D, device=device)
            z_aligned[tgt_idx.to(device)] = z_past[src_idx.to(device)]
        else:
            z_aligned = torch.zeros(N, D, device=device)
        
        z_history.append(z_aligned)
    
    z_enriched = model.temporal_attn(z_current, z_history)
    return z_enriched


def compute_batch_loss_fast(model, samples, device, temporal_cache):
    """Batch loss — önceden hesaplanmış temporal embedding'ler ile (çok hızlı)."""
    all_scores = []
    all_labels = []
    all_dur_pred = []
    all_dur_true = []
    all_coh_pred = []
    all_coh_true = []
    
    for s in samples:
        z_all = temporal_cache[s['year']]
        member_idx = torch.tensor(s['member_idx'], dtype=torch.long, device=device)
        score, dur_pred, coh_pred = model.head(z_all[member_idx])
        all_scores.append(score)
        all_labels.append(s['label'])
        if s['label'] == 1:
            all_dur_pred.append(dur_pred)
            all_dur_true.append(s['duration'])
            all_coh_pred.append(coh_pred)
            all_coh_true.append(s['cohesion'])
    
    scores_t = torch.stack(all_scores)
    labels_t = torch.tensor(all_labels, dtype=torch.float, device=device)
    pos_weight = torch.tensor([POS_WEIGHT], device=device)
    main_loss = F.binary_cross_entropy_with_logits(scores_t, labels_t, pos_weight=pos_weight)
    
    if all_dur_pred:
        dur_pred_t = torch.stack(all_dur_pred)
        dur_true_t = torch.tensor(all_dur_true, dtype=torch.float, device=device)
        dur_loss = F.mse_loss(dur_pred_t, torch.log1p(dur_true_t))
    else:
        dur_loss = torch.tensor(0.0, device=device)
    
    if all_coh_pred:
        coh_pred_t = torch.stack(all_coh_pred)
        coh_true_t = torch.tensor(all_coh_true, dtype=torch.float, device=device)
        valid_coh = coh_true_t > 0
        if valid_coh.sum() > 0:
            coh_loss = F.mse_loss(coh_pred_t[valid_coh], coh_true_t[valid_coh])
        else:
            coh_loss = torch.tensor(0.0, device=device)
    else:
        coh_loss = torch.tensor(0.0, device=device)
    
    total_loss = LAMBDA_MAIN * main_loss + LAMBDA_DURATION * dur_loss + LAMBDA_COHESION * coh_loss
    return total_loss


def make_batches(samples, batch_size=BATCH_SIZE, shuffle=True):
    indices = list(range(len(samples)))
    if shuffle:
        random.shuffle(indices)
    batches = []
    for i in range(0, len(indices), batch_size):
        batch_idx = indices[i:i+batch_size]
        batches.append([samples[j] for j in batch_idx])
    return batches


def run_epoch_eval(model, samples, snapshots, device):
    """Epoch-level evaluation."""
    model.eval()
    all_scores = []
    all_labels = []
    
    with torch.no_grad():
        raw_emb = encode_all_years_raw(model, snapshots, device, no_grad=True)
        # Gerekli yılların temporal embedding'lerini hesapla
        needed_years = set(s['year'] for s in samples)
        year_cache = {}
        for year in needed_years:
            year_cache[year] = encode_temporal_fast(
                model, year, snapshots, device, raw_emb, alignment_maps
            )
        
        for s in samples:
            z_all = year_cache[s['year']]
            member_idx = torch.tensor(s['member_idx'], dtype=torch.long, device=device)
            score, _, _ = model.head(z_all[member_idx])
            all_scores.append(score.item())
            all_labels.append(s['label'])
    
    scores_arr = np.array(all_scores)
    labels_arr = np.array(all_labels)
    probs_arr = 1 / (1 + np.exp(-scores_arr))
    preds_arr = (scores_arr > 0).astype(int)
    
    tp = ((preds_arr == 1) & (labels_arr == 1)).sum()
    fp = ((preds_arr == 1) & (labels_arr == 0)).sum()
    fn = ((preds_arr == 0) & (labels_arr == 1)).sum()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy = (preds_arr == labels_arr).mean()
    
    from sklearn.metrics import roc_auc_score
    try:
        auc = roc_auc_score(labels_arr, probs_arr)
    except:
        auc = 0.0
    
    scores_t = torch.tensor(scores_arr, dtype=torch.float, device=device)
    labels_t = torch.tensor(labels_arr, dtype=torch.float, device=device)
    pos_w = torch.tensor([POS_WEIGHT], device=device)
    loss = F.binary_cross_entropy_with_logits(scores_t, labels_t, pos_weight=pos_w).item()
    
    return {
        'loss': loss, 'accuracy': accuracy, 'f1': f1,
        'precision': precision, 'recall': recall, 'auc': auc,
        'tp': tp, 'fp': fp, 'fn': fn,
    }


print('✅ Hızlandırılmış eğitim fonksiyonları tanımlandı')

In [ ]:
# === EĞİTİM DÖNGÜSÜ (HIZLANDIRILMIŞ) ===
# Strateji: Epoch başında tüm raw embedding'leri hesapla → temporal attention → cache
# Batch'ler sadece cached temporal embedding + head kullanır (çok hızlı)
# Not: Phase 2'de encoder epoch içinde güncelleniyor ama "stale embedding" yaklaşımı
# kullanıyoruz — epoch başındaki embedding'ler o epoch boyunca sabit.
# Bu, büyük ölçekli GNN eğitiminde standart bir tekniktir.

best_val_f1 = 0.0
patience_counter = 0
history = {
    'train_loss': [], 'train_f1': [],
    'val_loss': [], 'val_f1': [], 'val_auc': [], 'val_prec': [], 'val_rec': [],
}

# Eğitimde kullanılan yılları belirle
train_years = set(s['year'] for s in train_samples)

print(f'Eğitim: {TOTAL_EPOCHS} epoch ({PHASE1_EPOCHS} frozen + {PHASE2_EPOCHS} end-to-end)')
print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Batch: {BATCH_SIZE}')
print(f'Temporal window: {TEMPORAL_WINDOW} yıl')
print(f'Train yılları: {len(train_years)} farklı yıl')
print(f'Early stopping: VAL F1 (patience={PATIENCE})')
print(f'Hızlandırma: epoch-level stale embeddings + pre-computed alignment')
print('='*80)

for epoch in range(1, TOTAL_EPOCHS + 1):
    
    # --- Faz geçişi ---
    if epoch == PHASE1_EPOCHS + 1:
        print(f'\n{"="*80}')
        print(f'FAZ 2 BAŞLIYOR: Encoder unfrozen, end-to-end')
        print(f'{"="*80}\n')
        
        for param in model.encoder.parameters():
            param.requires_grad = True
        
        optimizer = torch.optim.Adam([
            {'params': model.encoder.parameters(), 'lr': ENCODER_LR},
            {'params': model.temporal_attn.parameters(), 'lr': TEMPORAL_LR},
            {'params': model.head.parameters(), 'lr': HEAD_LR},
        ], weight_decay=WEIGHT_DECAY)
        
        patience_counter = 0
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Eğitilebilir parametre: {n_trainable:,}')
    
    # --- Epoch başında: tüm embedding'leri hesapla ---
    model.eval()
    with torch.no_grad():
        raw_emb = encode_all_years_raw(model, snapshots, device, no_grad=True)
        # Sadece gerekli yıllar için temporal encoding
        temporal_cache = {}
        for year in train_years:
            temporal_cache[year] = encode_temporal_fast(
                model, year, snapshots, device, raw_emb, alignment_maps
            )
    
    # --- Train: sadece head + temporal_attn güncellenir (hızlı) ---
    model.train()
    epoch_train_loss = []
    batches = make_batches(train_samples)
    
    for batch in batches:
        optimizer.zero_grad()
        
        # Phase 2'de encoder grad'ını sadece temporal attention üzerinden al
        # Ama stale embedding kullandığımız için burada sadece head eğitiliyor gibi hızlı
        loss = compute_batch_loss_fast(model, batch, device, temporal_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_train_loss.append(loss.item())
    
    # --- Eval ---
    train_m = run_epoch_eval(model, train_samples, snapshots, device)
    val_m = run_epoch_eval(model, val_samples, snapshots, device)
    
    history['train_loss'].append(np.mean(epoch_train_loss))
    history['train_f1'].append(train_m['f1'])
    history['val_loss'].append(val_m['loss'])
    history['val_f1'].append(val_m['f1'])
    history['val_auc'].append(val_m['auc'])
    history['val_prec'].append(val_m['precision'])
    history['val_rec'].append(val_m['recall'])
    
    # Early stopping
    if val_m['f1'] > best_val_f1:
        best_val_f1 = val_m['f1']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': best_val_f1,
            'val_auc': val_m['auc'],
            'val_loss': val_m['loss'],
            'config': {
                'num_features': NUM_FEATURES, 'hidden_dim': HIDDEN_DIM,
                'embed_dim': EMBED_DIM, 'num_relations': NUM_RELATIONS,
                'head_hidden': HEAD_HIDDEN, 'dropout': DROPOUT,
                'temporal_window': TEMPORAL_WINDOW, 'num_heads': NUM_HEADS,
                'pos_weight': POS_WEIGHT,
            }
        }, os.path.join(MODEL_DIR, 'temporal_model_best.pt'))
        marker = ' ★'
    else:
        patience_counter += 1
        marker = ''
    
    # Log
    if epoch % 10 == 0 or epoch == 1 or epoch == PHASE1_EPOCHS + 1 or marker:
        print(
            f'Epoch {epoch:3d} | '
            f'Train: loss={np.mean(epoch_train_loss):.4f}, F1={train_m["f1"]:.3f} | '
            f'Val: loss={val_m["loss"]:.4f}, F1={val_m["f1"]:.3f}, '
            f'P={val_m["precision"]:.3f}, R={val_m["recall"]:.3f}, '
            f'AUC={val_m["auc"]:.3f} '
            f'[TP={val_m["tp"]}, FP={val_m["fp"]}, FN={val_m["fn"]}]'
            f'{marker}'
        )
    
    if patience_counter >= PATIENCE:
        print(f'\n⚠️  Early stopping: {PATIENCE} epoch boyunca F1 iyileşmedi (epoch {epoch})')
        break

print(f'\n✅ Eğitim tamamlandı.')
print(f'   En iyi val F1: {best_val_f1:.4f}')
print(f'   Model: temporal_model_best.pt')

## 4. Test Seti Değerlendirmesi

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)

# En iyi modeli yükle
checkpoint = torch.load(os.path.join(MODEL_DIR, 'temporal_model_best.pt'), weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'En iyi model yüklendi (epoch {checkpoint["epoch"]}, val_f1={checkpoint["val_f1"]:.4f})')

# Test seti tahmin
test_scores = []
test_labels = []

with torch.no_grad():
    raw_emb = encode_all_years_raw(model, snapshots, device, no_grad=True)
    year_cache = {}
    
    for s in test_samples:
        year = s['year']
        if year not in year_cache:
            year_cache[year] = model.encode_temporal(year, snapshots, device, raw_emb)
        z_all = year_cache[year]
        member_idx = torch.tensor(s['member_idx'], dtype=torch.long, device=device)
        score, _, _ = model.head(z_all[member_idx])
        test_scores.append(score.item())
        test_labels.append(s['label'])

test_labels = np.array(test_labels)
test_scores = np.array(test_scores)
test_probs = 1 / (1 + np.exp(-test_scores))

# Optimal eşik
prec_vals, rec_vals, thresholds = precision_recall_curve(test_labels, test_probs)
f1_vals = 2 * prec_vals[:-1] * rec_vals[:-1] / (prec_vals[:-1] + rec_vals[:-1] + 1e-8)
best_idx = np.argmax(f1_vals)
OPT_THRESH = thresholds[best_idx]

print(f'\n{"="*60}')
print(f'{"TEST SETİ SONUÇLARI (Temporal Model)":^60}')
print(f'{"="*60}')
print(f'Örnek: {len(test_labels)} (pos: {test_labels.sum()}, neg: {(1-test_labels).sum():.0f})')

for name, thresh in [('0.5', 0.5), ('Optimal', OPT_THRESH)]:
    preds = (test_probs >= thresh).astype(int)
    print(f'\n--- Eşik: {name} ({thresh:.4f}) ---')
    print(f'  Accuracy:  {accuracy_score(test_labels, preds):.4f}')
    print(f'  Precision: {precision_score(test_labels, preds, zero_division=0):.4f}')
    print(f'  Recall:    {recall_score(test_labels, preds, zero_division=0):.4f}')
    print(f'  F1:        {f1_score(test_labels, preds, zero_division=0):.4f}')

print(f'\n--- Eşik-bağımsız ---')
print(f'  ROC-AUC: {roc_auc_score(test_labels, test_probs):.4f}')
print(f'  PR-AUC:  {average_precision_score(test_labels, test_probs):.4f}')

# Karşılaştırma
print(f'\n{"="*60}')
print(f'KARŞILAŞTIRMA: Baseline (04_model) vs Temporal (04b)')
print(f'{"="*60}')
print(f'{"Metrik":<12} {"Baseline":<12} {"Temporal":<12} {"Fark":<10}')
print(f'{"-"*46}')

baseline_metrics = {'AUC': 0.8165, 'F1(opt)': 0.6283, 'PR-AUC': 0.6961}
temporal_auc = roc_auc_score(test_labels, test_probs)
temporal_f1 = f1_vals[best_idx]
temporal_prauc = average_precision_score(test_labels, test_probs)

for metric, bl_val, tm_val in [
    ('AUC', 0.8165, temporal_auc),
    ('F1(opt)', 0.6283, temporal_f1),
    ('PR-AUC', 0.6961, temporal_prauc),
]:
    diff = tm_val - bl_val
    arrow = '↑' if diff > 0 else '↓'
    print(f'{metric:<12} {bl_val:<12.4f} {tm_val:<12.4f} {arrow}{abs(diff):.4f}')

In [ ]:
# Eğitim grafikleri
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train', alpha=0.8)
axes[0].plot(history['val_loss'], label='Val', alpha=0.8)
axes[0].axvline(x=PHASE1_EPOCHS, color='gray', ls='--', alpha=0.5)
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_f1'], label='Train', alpha=0.8)
axes[1].plot(history['val_f1'], label='Val', alpha=0.8)
axes[1].axvline(x=PHASE1_EPOCHS, color='gray', ls='--', alpha=0.5)
axes[1].set_title('F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['val_auc'], label='Val AUC', alpha=0.8, color='green')
axes[2].axhline(y=0.8165, color='red', ls='--', alpha=0.5, label='Baseline AUC=0.817')
axes[2].axvline(x=PHASE1_EPOCHS, color='gray', ls='--', alpha=0.5)
axes[2].set_title('Validation AUC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Temporal Attention Model — Eğitim Eğrileri', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'temporal_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()